In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import joblib
from google.colab import files

# 1. Load data
matches = pd.read_csv("international_matches_transformed.csv")
matches["date"] = pd.to_datetime(matches["date"])

# 2. Rolling Averages & H2H
cols = ["gf", "ga"]
new_cols = [f"{c}_rolling" for c in cols]

def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    return group.dropna(subset=new_cols)

matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))
matches_rolling = matches_rolling.droplevel('team')

def get_h2h_win_rate(group):
    group = group.sort_values("date")
    h2h_rate = group['target'].expanding().mean().shift(1)
    group['h2h_win_rate'] = h2h_rate.fillna(0.5)
    return group

matches_rolling['matchup'] = matches_rolling.apply(lambda x: "-".join(sorted([x['team'], x['opponent']])), axis=1)
matches_rolling = matches_rolling.groupby(['team', 'matchup']).apply(get_h2h_win_rate)
matches_rolling = matches_rolling.droplevel(['team', 'matchup'])

# 3. Weights
def calculate_weights(row):
    tourney_weights = {'FIFA World Cup': 5.0, 'UEFA Euro': 3.5, 'Copa América': 3.5, 'Friendly': 1.0}
    t_weight = tourney_weights.get(row['tournament'], 2.0)
    years_ago = 2026 - row['date'].year
    time_weight = np.exp(-0.05 * years_ago)
    return t_weight * time_weight

matches_rolling['sample_weight'] = matches_rolling.apply(calculate_weights, axis=1)

# 4. Train on the FULL dataset
predictors = ["venue_code", "opp_code", "day_code", "h2h_win_rate"]
all_features = predictors + new_cols

rf_final = RandomForestClassifier(n_estimators=100, min_samples_split=10, random_state=1)
rf_final.fit(matches_rolling[all_features], matches_rolling["target"], sample_weight=matches_rolling['sample_weight'])

# 5. Export Lookups for the API
# Team Mapping
team_mapping = matches_rolling[['opponent', 'opp_code']].drop_duplicates().sort_values('opp_code')
team_mapping.to_csv("team_mapping.csv", index=False)

# Latest Form (The most recent stats for every team)
latest_form = matches_rolling.groupby("team").tail(1)[["team", "gf_rolling", "ga_rolling"]]
latest_form.to_csv("latest_form.csv", index=False)

# H2H History (The most recent win rate for every specific rivalry)
h2h_lookup = matches_rolling.groupby("matchup").tail(1)[["matchup", "h2h_win_rate"]]
h2h_lookup.to_csv("h2h_lookup.csv", index=False)

# 6. Exporting the Model
joblib.dump(rf_final, "world_cup_model.joblib")

# Download for my Docker 'data/' folder
for f in ["world_cup_model.joblib", "team_mapping.csv", "latest_form.csv", "h2h_lookup.csv"]:
    files.download(f)


/tmp/ipython-input-2469750801.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))
/tmp/ipython-input-2469750801.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = matches_rolling.groupby(['team', 'matchup']).apply(get_h2h_win_rate)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>